# 03 — Quant Feature Engineering

1. Setup y reconstrucción del split temporal
2. Features de trayectoria
3. Features de volatilidad
4. Features de dinámica intradía
5. Dataset final de features

In [9]:
import pandas as pd
import numpy as np

input_train = pd.read_csv("../data/input_training.csv")
output_train = pd.read_csv("../data/output_training_gmEd6Zt.csv")

input_test = pd.read_csv("../data/input_test.csv")
output_test = pd.read_csv("../data/output_test_random.csv")

return_cols = [f"r{i}" for i in range(53)]

train = input_train.merge(
    output_train,
    on="ID",
    how="inner",
    validate="one_to_one"
)

In [10]:
# Reconstruimos exactamente el split decidido en 02:
# Split temporal definido en 02_validation_and_preprocessing
TRAIN_END_DAY = 401
VAL_START_DAY = 402

train_dev = train.loc[train["day"] <= TRAIN_END_DAY].copy()
val_dev = train.loc[train["day"] >= VAL_START_DAY].copy()
print("Train observations:", len(train_dev))
print("Validation observations:", len(val_dev))
print("Train day range:",train_dev["day"].min(),"-",train_dev["day"].max())
print("Validation day range:",val_dev["day"].min(),"-",val_dev["day"].max())

Train observations: 673751
Validation observations: 169548
Train day range: 0 - 401
Validation day range: 402 - 502


## 2. Features de trayectoria intradía

La exploración realizada en `01_research_hypotheses.ipynb` mostró que el retorno
acumulado contiene una relación no lineal con el régimen futuro: trayectorias con
desplazamientos intradía extremos presentan una mayor probabilidad de terminar en
un régimen direccional, mientras que trayectorias cercanas a cero presentan una
mayor frecuencia de la clase neutral.

También observamos que esta estructura aparece en distintas partes de la sesión,
por lo que resulta útil conservar información sobre cómo evoluciona el desplazamiento
a lo largo del día.

En esta sección transformamos la secuencia de 53 retornos en un conjunto compacto
de características de trayectoria.

Las features se calculan sobre los retornos originales expresados en basis points
para preservar su interpretación financiera.

### 2.1 Retorno acumulado

Para el *feature engineering* de horizontes intradía, consideraremos lo siguiente:

* **Tratamiento de NaN**: Calculamos el retorno usando únicamente valores observados (sin imputar ceros antes).
* **Control de Outliers**: Evitamos el producto compuesto y la suma de log-retornos, ya que existen valores anómalos inferiores al -100%.

Utilizaremos como aproximación de desplazamiento (*path*):

$$R_{\text{path}} \approx \sum_{t=0}^{52} r_t$$


In [11]:
def row_sum_observed(df, cols):
    return df[cols].sum(axis=1, skipna=True)


for df in [train_dev, val_dev]:
    df["path_return_bps"] = row_sum_observed(df, return_cols)

#2.2 Trayectoria por ventanas
#Recuperamos la división que utilizamos en H2:
early_cols = [f"r{i}" for i in range(0, 18)]
middle_cols = [f"r{i}" for i in range(18, 36)]
late_cols = [f"r{i}" for i in range(36, 53)]
for df in [train_dev, val_dev]:

    df["return_early_bps"] = row_sum_observed(df, early_cols)
    df["return_middle_bps"] = row_sum_observed(df, middle_cols)
    df["return_late_bps"] = row_sum_observed(df, late_cols)

### 2.2 Segmentación de la trayectoria

* **Qué se hace**: Calculamos retornos acumulados por tramos del día: $R_{\text{early}}$, $R_{\text{middle}}$ y $R_{\text{late}}$.
* **Por qué**: Permite al modelo diferenciar la dirección del camino. Por ejemplo, distingue $(+, +, +)$ de $(+, +, -)$, aunque ambos terminen con un desplazamiento final similar.

### 2.3 Magnitud del desplazamiento

* **Qué se hace**: Incluimos explícitamente el valor absoluto del desplazamiento total: $|R_{\text{path}}|$.
* **Por qué**: Captura relaciones no lineales. Los extremos de los retornos acumulados (grandes subidas o grandes caídas) presentan mayor probabilidad de activar un régimen direccional.


In [12]:
for df in [train_dev, val_dev]:
    df["abs_path_return_bps"] = df["path_return_bps"].abs()
#Esto separa dos conceptos:

#path_return_bps      → dirección del desplazamiento
#abs_path_return_bps  → intensidad del desplazamiento

### 2.4 Cambio entre inicio y final de la trayectoria

* **Qué se hace**: Calculamos una única variable de evolución temporal: 
$$\Delta R = R_{\text{late}} - R_{\text{early}}$$
* **Por qué**: Sustituye y simplifica diez variables por una sola, capturando de forma directa el cambio neto en la tendencia intradía.


In [13]:
for df in [train_dev, val_dev]:
    df["early_late_change_bps"] = (
        df["return_late_bps"]
        - df["return_early_bps"]
    )
path_features = [
    "path_return_bps",
    "abs_path_return_bps",
    "return_early_bps",
    "return_middle_bps",
    "return_late_bps",
    "early_late_change_bps",
]

train_dev[path_features].describe().T

,count,mean,std,min,25%,50%,75%,max
path_return_bps,673751.0,1047.193482,544780.771046,-1.987691e+04,-107.73,-6.21,79.69,4.310702e+08
abs_path_return_bps,673751.0,1249.431252,544780.344760,0.000000e+00,38.72,92.98,192.90,4.310702e+08
return_early_bps,673751.0,1050.800577,544780.719419,-1.989236e+04,-81.57,-1.93,61.67,4.310702e+08
return_middle_bps,673751.0,-2.344122,101.645075,-4.096710e+03,-41.14,0.00,37.65,6.699440e+03
return_late_bps,673751.0,-1.262972,79.871266,-3.140540e+03,-30.60,0.00,28.76,5.230770e+03
early_late_change_bps,673751.0,-1052.063549,544780.742871,-4.310702e+08,-70.38,2.63,89.10,1.990870e+04


In [14]:
#2.1 Crear versión robusta de los retornos originales
#Usaría 0.1%–99.9% por intervalo. Es suficientemente conservador y ya vimos esos cuantiles durante el diagnóstico.
# Límites estimados exclusivamente sobre train
lower_bounds = train_dev[return_cols].quantile(0.001)
upper_bounds = train_dev[return_cols].quantile(0.999)
def clip_raw_returns(df, return_cols, lower, upper):
    out = df.copy()

    out[return_cols] = out[return_cols].clip(
        lower=lower,
        upper=upper,
        axis=1
    )

    return out


train_feat = clip_raw_returns(
    train_dev,
    return_cols,
    lower_bounds,
    upper_bounds
)

val_feat = clip_raw_returns(
    val_dev,
    return_cols,
    lower_bounds,
    upper_bounds
)
# calculamos las mismas 6 features
def add_path_features(df):

    df = df.copy()

    df["path_return_bps"] = df[return_cols].sum(axis=1, skipna=True)

    df["return_early_bps"] = df[early_cols].sum(axis=1, skipna=True)
    df["return_middle_bps"] = df[middle_cols].sum(axis=1, skipna=True)
    df["return_late_bps"] = df[late_cols].sum(axis=1, skipna=True)

    df["abs_path_return_bps"] = df["path_return_bps"].abs()

    df["early_late_change_bps"] = (
        df["return_late_bps"]
        - df["return_early_bps"]
    )

    return df


train_feat = add_path_features(train_feat)
val_feat = add_path_features(val_feat)
train_feat[path_features].describe().T

,count,mean,std,min,25%,50%,75%,max
path_return_bps,673751.0,-33.468035,292.186858,-3491.96809,-107.52,-6.33,79.35,4185.64313
abs_path_return_bps,673751.0,164.155974,244.020587,0.00000,38.64,92.70,191.23,4185.64313
return_early_bps,673751.0,-29.811015,265.006347,-3437.91677,-81.54,-1.97,61.53,3000.45530
return_middle_bps,673751.0,-2.373711,92.840922,-1552.37245,-41.10,0.00,37.57,2040.33730
return_late_bps,673751.0,-1.283308,71.961493,-1124.35735,-30.57,0.00,28.70,1546.85292
early_late_change_bps,673751.0,28.527707,273.466402,-3097.84761,-70.16,2.68,88.85,3571.48609


### Conclusiones clave del análisis

* **Eliminación de ruido**: Desapareció la contaminación numérica; los extremos ahora están en una escala económicamente interpretable.
* **Estructura de volatilidad**: El inicio de la sesión concentra la mayor dispersión, decreciendo a lo largo del día:
$$\sigma_{\text{early}} = 265 > \sigma_{\text{middle}} = 93 > \sigma_{\text{late}} = 72$$
* **Asimetría del desplazamiento**: El sesgo a la baja ocurre exclusivamente al principio de la sesión, facilitando la detección de *momentum*, reversión o neutralidad:
$$\text{median}(R_{\text{path}}) = -6.33 \quad \text{vs.} \quad \text{median}(R_{\text{early}}) \approx -1.97, \; \text{median}(R_{\text{middle}}) = 0, \; \text{median}(R_{\text{late}}) = 0$$


### 3.0 Definir los retornos que alimentan las features
Si el DataFrame donde ya aplicaste imputación + robust scaling + clipping es X_train_baseline, entonces:


## 3. Volatilidad y actividad de la trayectoria

Las features anteriores resumen el desplazamiento direccional de la trayectoria. Sin embargo, dos trayectorias con un retorno acumulado similar pueden haber seguido dinámicas intradía muy diferentes.

Por ello incorporamos medidas de actividad independientes de la dirección:

- **Realized volatility:** magnitud cuadrática de los movimientos intradía.
- **Mean absolute return:** magnitud típica de los movimientos de 5 minutos.
- **Early volatility share:** proporción de la actividad total concentrada en la primera parte de la sesión.

Estas variables buscan capturar no sólo cuánto se desplazó el precio, sino también cómo se distribuyó la actividad durante la ventana observada.

In [18]:
# 3.1 realized_vol_bps
# Como nuestros r0,...,r52 ya están expresados en bps:

# Realized volatility de la trayectoria
train_feat["realized_vol_bps"] = np.sqrt(
    (train_feat[return_cols] ** 2).sum(axis=1, skipna=True)
)

val_feat["realized_vol_bps"] = np.sqrt(
    (val_feat[return_cols] ** 2).sum(axis=1, skipna=True)
)
train_feat["realized_vol_bps"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)

count    673751.000000
mean        194.379075
std         216.936893
min           0.000000
50%         140.393075
75%         219.068160
90%         350.012985
95%         481.243293
99%        1333.768890
max        2627.706886
Name: realized_vol_bps, dtype: float64

### Actividad de la trayectoria

* **Fórmula**: 
$$RV_i = \sqrt{\sum_{t=1}^{53} r_{i,t}^2}$$
* **Concepto**: No representa volatilidad anualizada; funciona estrictamente como una métrica de actividad realizada dentro de la trayectoria observada.

### Volatilidad ajustada

* **Fórmula**:
$$RV_i^{\text{adj}} = \sqrt{\frac{53}{n_i} \sum_{t \in \text{observed}} r_{i,t}^2}$$
* **Concepto**: Escala la actividad realizada multiplicando por la proporción de datos faltantes, corrigiendo el sesgo cuando la trayectoria tiene datos omitidos ($n_i$ representa el número de retornos observados).




In [19]:
#Ajustar la volatilidad por disponibilidad
#Como todas las trayectorias deberían representar 53 intervalos,
# podemos calcular primero el número de retornos observados:
train_feat["n_obs"] = train_feat[return_cols].notna().sum(axis=1)
val_feat["n_obs"] = val_feat[return_cols].notna().sum(axis=1)
train_feat["realized_vol_adj_bps"] = np.where(train_feat["n_obs"] > 0,np.sqrt(
    (53 / train_feat["n_obs"])* (train_feat[return_cols] ** 2).sum(axis=1, skipna=True)),np.nan)

val_feat["realized_vol_adj_bps"] = np.where(val_feat["n_obs"] > 0,np.sqrt(
        (53 / val_feat["n_obs"])* (val_feat[return_cols] ** 2).sum(axis=1, skipna=True)),
    np.nan)
#Esto no reconstruye los retornos faltantes. Simplemente hace comparable la escala de actividad 
# entre trayectorias con diferente número de observaciones, bajo una aproximación de que los retornos 
# disponibles son representativos de la intensidad de la trayectoria.
train_feat[["n_obs", "realized_vol_bps", "realized_vol_adj_bps"]].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99])

,n_obs,realized_vol_bps,realized_vol_adj_bps
count,673751.000000,673751.000000,672072.000000
mean,47.289200,194.379075,213.834448
std,12.236509,216.936893,251.879709
min,0.000000,0.000000,0.000000
50%,53.000000,140.393075,148.376782
75%,53.000000,219.068160,236.782387
90%,53.000000,350.012985,394.379383
95%,53.000000,481.243293,551.631053
99%,53.000000,1333.768890,1514.187265
max,53.000000,2627.706886,14307.745568


### Validación en Holdout

* **Antecedente**: En la sección *01* confirmamos que una mayor volatilidad está fuertemente asociada con la probabilidad del régimen direccional:
$$P(|r_{\text{eod}}| = 1)$$
* **Objetivo**: Comprobar si esta relación se mantiene en el *holdout* temporal utilizando la variable ya limpia y ajustada ($RV_i^{\text{adj}}$).


In [ ]:
train_feat["directional"] = (train_feat["reod"].abs() == 1).astype(int)
val_feat["directional"] = (val_feat["reod"].abs() == 1).astype(int)
q20, q40, q60, q80 = train_feat["realized_vol_adj_bps"].quantile([0.20, 0.40, 0.60, 0.80])
vol_bins = [
    -np.inf,
    q20,
    q40,
    q60,
    q80,
    np.inf
]

vol_labels = ["Q1", "Q2", "Q3", "Q4", "Q5"]

train_feat["vol_quintile"] = pd.cut(
    train_feat["realized_vol_adj_bps"],
    bins=vol_bins,
    labels=vol_labels
)

val_feat["vol_quintile"] = pd.cut(
    val_feat["realized_vol_adj_bps"],
    bins=vol_bins,
    labels=vol_labels
)

In [22]:
vol_signal_check = pd.DataFrame({
    "train": (
        train_feat
        .groupby("vol_quintile", observed=True)["directional"]
        .mean()
    ),
    "validation": (
        val_feat
        .groupby("vol_quintile", observed=True)["directional"]
        .mean()
    )
})

vol_signal_check

,train,validation
vol_quintile,,
Q1,0.405952,0.380460
Q2,0.577053,0.560826
Q3,0.644613,0.633511
Q4,0.688098,0.679802
Q5,0.643775,0.636724


### 3.1 Resultados de la validación

* **Conclusión principal**: La volatilidad intradía contiene una señal predictiva fuerte y generaliza correctamente en el tiempo. La relación es estable ya que los resultados de *validation* reproducen casi exactamente los de *train*.
* **Impacto estadístico**: 
  * Baja volatilidad: $P(|r_{\text{eod}}| = 1) \approx 38\%$
  * Alta volatilidad (Q4): $P(|r_{\text{eod}}| = 1) \approx 68\%$
* **Efecto no monótono**: Se descubrió que en el quintil extremo (Q5) la probabilidad cae respecto a Q4. 
* **Interpretación**: Más volatilidad ayuda a identificar el régimen direccional solo hasta cierto límite; la volatilidad extrema deja de aportar señal y probablemente representa un régimen de mercado distinto.


### 2.6 Eficiencia de la trayectoria

* **Concepto**: Mide si el movimiento intradía (hasta las 14:00) fue direccional o si tuvo fluctuaciones de ida y vuelta.
* **Fórmula**:
$$\text{Efficiency} = \frac{|R_{\text{path}}|}{\sum_{t=0}^{52} |r_t|}$$
* **Interpretación**:
  * **Cerca de 1**: Trayectoria limpia y puramente direccional.
  * **Cerca de 0**: Mucho ruido y volatilidad, pero con un desplazamiento neto mínimo.


In [23]:
# Ya tenemos abs_path_return_bps, así que sólo necesitamos calcular la distancia total recorrida:
# Eficiencia de la trayectoria

train_feat["total_abs_move_bps"] = (
    train_feat[return_cols].abs().sum(axis=1, skipna=True)
)

val_feat["total_abs_move_bps"] = (
    val_feat[return_cols].abs().sum(axis=1, skipna=True)
)

train_feat["path_efficiency"] = np.where(
    train_feat["total_abs_move_bps"] > 0,
    train_feat["abs_path_return_bps"] / train_feat["total_abs_move_bps"],
    0.0
)

val_feat["path_efficiency"] = np.where(
    val_feat["total_abs_move_bps"] > 0,
    val_feat["abs_path_return_bps"] / val_feat["total_abs_move_bps"],
    0.0
)

train_feat["path_efficiency"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

count    673751.000000
mean          0.219727
std           0.211927
min           0.000000
25%           0.073092
50%           0.161240
75%           0.289486
90%           0.481035
95%           0.688409
99%           1.000000
max           1.000000
Name: path_efficiency, dtype: float64

no tenemos el problema de outliers que tuvimos anteriormente y existe suficiente dispersión para distinguir trayectorias muy ruidosas de trayectorias direccionales.

Pero todavía no sabemos si sirve para predecir reod.

Hagamos exactamente la misma prueba temporal que con volatilidad: quintiles definidos en train y aplicados sin recalcular a validation.

In [24]:
# Cortes aprendidos exclusivamente en train
eff_cuts = train_feat["path_efficiency"].quantile(
    [0.20, 0.40, 0.60, 0.80]
).values

eff_bins = [-np.inf, *eff_cuts, np.inf]
eff_labels = ["Q1", "Q2", "Q3", "Q4", "Q5"]

train_feat["eff_quintile"] = pd.cut(
    train_feat["path_efficiency"],
    bins=eff_bins,
    labels=eff_labels
)

val_feat["eff_quintile"] = pd.cut(
    val_feat["path_efficiency"],
    bins=eff_bins,
    labels=eff_labels
)

eff_signal_check = pd.DataFrame({
    "train": train_feat.groupby(
        "eff_quintile", observed=True
    )["directional"].mean(),

    "validation": val_feat.groupby(
        "eff_quintile", observed=True
    )["directional"].mean()
})

eff_signal_check

,train,validation
eff_quintile,,
Q1,0.576107,0.571615
Q2,0.620438,0.604197
Q3,0.622998,0.611653
Q4,0.626857,0.608325
Q5,0.505766,0.494117


### 3.2 Análisis de Eficiencia (`path_efficiency`)

* **Decisión**: Se mantiene la variable en el modelo.
* **Justificación**: No se debe a una relación lineal, sino al descubrimiento de un patrón no lineal y persistente que generaliza correctamente:
  * **Eficiencia moderada**: $P(|r_{\text{eod}}| = 1) \approx 61\%$
  * **Eficiencia extrema**: $P(|r_{\text{eod}}| = 1) \approx 49\%$
* **Hipótesis estadística**: Los movimientos intradía demasiado limpios (extremos) muestran una caída en la probabilidad direccional. Esto sugiere un posible efecto de agotamiento o reversión, aunque por ahora solo se valida el patrón estadístico sin asegurar el mecanismo económico exacto.

realized_vol_bps     → SE QUEDA

path_efficiency      → SE QUEDA

###  Concentración temporal de la volatilidad

Ya sabemos que `realized_vol_bps` importa. Ahora queremos saber **cuándo** ocurrió esa volatilidad. 

La feature será:

$$EarlyVolShare = \frac{\sum_{t \in Early} r_t^2}{\sum_{t=0}^{52} r_t^2}$$

* **Cerca de 1** → Gran parte de la actividad ocurrió temprano.
* **Cerca de 0** → La actividad se concentró más adelante.

Usamos directamente `train_feat` y `val_feat`, que contienen los retornos winsorizados.


In [25]:
# Proporción de volatilidad concentrada en la ventana early

def add_early_vol_share(df):
    total_var = (df[return_cols] ** 2).sum(axis=1, skipna=True)
    early_var = (df[early_cols] ** 2).sum(axis=1, skipna=True)

    df["early_vol_share"] = np.where(
        total_var > 0,
        early_var / total_var,
        0.0
    )
    return df


train_feat = add_early_vol_share(train_feat)
val_feat = add_early_vol_share(val_feat)

train_feat["early_vol_share"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

count    673751.000000
mean          0.612101
std           0.222171
min           0.000000
25%           0.493005
50%           0.638008
75%           0.763056
90%           0.872153
95%           0.953810
99%           1.000000
max           1.000000
Name: early_vol_share, dtype: float64

In [26]:
# Cortes aprendidos sólo en train
early_vol_cuts = train_feat["early_vol_share"].quantile(
    [0.20, 0.40, 0.60, 0.80]
).values

early_vol_bins = [-np.inf, *early_vol_cuts, np.inf]
labels = ["Q1", "Q2", "Q3", "Q4", "Q5"]

train_feat["early_vol_quintile"] = pd.cut(
    train_feat["early_vol_share"],
    bins=early_vol_bins,
    labels=labels
)

val_feat["early_vol_quintile"] = pd.cut(
    val_feat["early_vol_share"],
    bins=early_vol_bins,
    labels=labels
)

early_vol_check = pd.DataFrame({
    "train": train_feat.groupby(
        "early_vol_quintile", observed=True
    )["directional"].mean(),

    "validation": val_feat.groupby(
        "early_vol_quintile", observed=True
    )["directional"].mean()
})

early_vol_check

,train,validation
early_vol_quintile,,
Q1,0.527328,0.518943
Q2,0.628223,0.615341
Q3,0.624423,0.612386
Q4,0.617907,0.605635
Q5,0.554286,0.538668


### Análisis de Early Vol Share

* **Concentración intermedia:** $P(|r_{eod}| = 1) \approx 61\%$
* **Extremos:** Tienen menor probabilidad direccional.

Por lo tanto, no es simplemente *"más volatilidad temprano = más direccionalidad futura"*.

SE QUEDA

`early_vol_share` aporta una dimensión diferente de `realized_vol_bps`:
* `realized_vol_bps` → Cuánta actividad hubo.
* `early_vol_share` → Cuándo ocurrió esa actividad.

>  **Conclusión clave:** El patrón train/validation es prácticamente idéntico, que es justo lo que queríamos comprobar.


### Actividad reciente

La hipótesis es:

No sólo importa cuánta volatilidad hubo, sino cuánto de ese movimiento ocurrió cerca de las 14:00.

Ya tenemos `late_cols`, así que construimos:

$$LateVolShare = \frac{\sum_{t \in Late} r_t^2}{\sum_{t=0}^{52} r_t^2}$$


In [27]:
# Proporción de volatilidad concentrada en la ventana late

def add_late_vol_share(df):
    total_var = (df[return_cols] ** 2).sum(axis=1, skipna=True)
    late_var = (df[late_cols] ** 2).sum(axis=1, skipna=True)

    df["late_vol_share"] = np.where(
        total_var > 0,
        late_var / total_var,
        0.0
    )

    return df


train_feat = add_late_vol_share(train_feat)
val_feat = add_late_vol_share(val_feat)

train_feat["late_vol_share"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

count    673751.000000
mean          0.148748
std           0.139067
min           0.000000
25%           0.063303
50%           0.116643
75%           0.192600
90%           0.297144
95%           0.390597
99%           0.751582
max           1.000000
Name: late_vol_share, dtype: float64

- análisis de la distribución marginal de una variable candidata antes de evaluar su relación con el target
Que tenga variabilidad. Si casi todas las observaciones fueran, por ejemplo, 0.10, la feature difícilmente discriminaría observaciones. Aquí tenemos:
$$ Q_{25}=0.063, \ Q_{50}=0.117, \ Q_{75} = 0.193 $$
- Hay dispersión clara.

- la feature pasó el sanity check de distribución, no porque haya demostrado valor predictivo.
- Lo único que sabemos ahora es que está bien definida y no parece degenerada.

evaluamos la relacion la relación $ P(∣reod∣=1∣LateVolShare) $ 
- Lo que queremos ver es muy concreto: si cambia entre quintiles y, sobre todo, si el patrón observado en train se reproduce en validation.

In [ ]:
# Validación temporal de late_vol_share

late_vol_cuts = train_feat["late_vol_share"].quantile(
    [0.20, 0.40, 0.60, 0.80]
).values

late_vol_bins = [-np.inf, *late_vol_cuts, np.inf]
labels = ["Q1", "Q2", "Q3", "Q4", "Q5"]

train_feat["late_vol_quintile"] = pd.cut(
    train_feat["late_vol_share"],
    bins=late_vol_bins,
    labels=labels
)

val_feat["late_vol_quintile"] = pd.cut(
    val_feat["late_vol_share"],
    bins=late_vol_bins,
    labels=labels
)

late_vol_check = pd.DataFrame({
    "train": train_feat.groupby(
        "late_vol_quintile", observed=True
    )["directional"].mean(),

    "validation": val_feat.groupby(
        "late_vol_quintile", observed=True
    )["directional"].mean()
})

late_vol_check

,train,validation
late_vol_quintile,,
Q1,0.477458,0.462509
Q2,0.611911,0.593821
Q3,0.622605,0.609337
Q4,0.638538,0.626015
Q5,0.601655,0.604123


la forma se reproduce temporalmente casi exactamente. Q1 tiene mucha menor probabilidad de régimen direccional; Q2–Q4 suben fuertemente; Q5 cae ligeramente. Las diferencias train-validation son pequeñas.

### Volatility concentration

Quiero medir qué tan concentrada está la volatilidad dentro de toda la trayectoria, no solo early/late.

**Idea:**

$$vol\_concentration = \frac{\max_{t}(r_t^2)}{\sum_{t} r_t^2}$$

* **Cerca de 0:** La volatilidad está distribuida entre muchos intervalos.
* **Alta:** Una parte importante del movimiento provino de uno o pocos shocks.


In [29]:
# Volatility concentration
sq_returns_train = train_feat[return_cols] ** 2
sq_returns_val = val_feat[return_cols] ** 2

train_feat["vol_concentration"] = (
    sq_returns_train.max(axis=1)
    / sq_returns_train.sum(axis=1)
)

val_feat["vol_concentration"] = (
    sq_returns_val.max(axis=1)
    / sq_returns_val.sum(axis=1)
)

train_feat["vol_concentration"] = (
    train_feat["vol_concentration"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

val_feat["vol_concentration"] = (
    val_feat["vol_concentration"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

train_feat["vol_concentration"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

count    673751.000000
mean          0.312972
std           0.219040
min           0.000000
25%           0.169255
50%           0.241952
75%           0.373039
90%           0.615551
95%           0.895926
99%           1.000000
max           1.000000
Name: vol_concentration, dtype: float64

- Mediana = 0.242 → en una trayectoria típica, el intervalo más violento explica aproximadamente 24% de toda la energía/varianza realizada.
- 75% = 0.373 → todavía razonable.
- 90% = 0.616 → en el 10% más concentrado, un solo intervalo explica más del 61%.
- 95% = 0.896 → aquí prácticamente toda la volatilidad proviene de un único shock.
- Tenemos todo el rango [0,1], por lo que la variable realmente distingue trayectorias muy diferentes.

In [30]:
# Quintiles definidos únicamente con train
vol_conc_bins = train_feat["vol_concentration"].quantile(
    [0, .2, .4, .6, .8, 1]
).values

vol_conc_bins[0] = -np.inf
vol_conc_bins[-1] = np.inf

train_feat["vol_conc_quintile"] = pd.cut(
    train_feat["vol_concentration"],
    bins=vol_conc_bins,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"]
)

val_feat["vol_conc_quintile"] = pd.cut(
    val_feat["vol_concentration"],
    bins=vol_conc_bins,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"]
)

pd.concat({
    "train": train_feat.groupby("vol_conc_quintile", observed=True)["directional"].mean(),
    "validation": val_feat.groupby("vol_conc_quintile", observed=True)["directional"].mean()
}, axis=1)

,train,validation
vol_conc_quintile,,
Q1,0.605242,0.595339
Q2,0.635043,0.622582
Q3,0.627911,0.610678
Q4,0.601039,0.589718
Q5,0.482931,0.476478


Cuando la volatilidad está extremadamente concentrada en un solo intervalo, la probabilidad de régimen direccional cae aproximadamente de 60–63% a 48%. Es decir:

Una trayectoria dominada por un shock aislado parece mucho menos compatible con un régimen direccional persistente.

### Sign Persistence

¿Qué proporción de los retornos intradía observados comparte el mismo signo que el retorno total de la trayectoria?

In [31]:
# Signo final de la trayectoria
path_sign_train = np.sign(train_feat["path_return_bps"])
path_sign_val = np.sign(val_feat["path_return_bps"])

# Signo de cada retorno intradía
signs_train = np.sign(train_feat[return_cols])
signs_val = np.sign(val_feat[return_cols])

# ¿Cada intervalo coincide con la dirección final?
same_sign_train = signs_train.eq(path_sign_train, axis=0)
same_sign_val = signs_val.eq(path_sign_val, axis=0)

# Solo contamos intervalos realmente observados
observed_train = train_feat[return_cols].notna()
observed_val = val_feat[return_cols].notna()

train_feat["sign_persistence"] = (
    (same_sign_train & observed_train).sum(axis=1)
    / observed_train.sum(axis=1)
)

val_feat["sign_persistence"] = (
    (same_sign_val & observed_val).sum(axis=1)
    / observed_val.sum(axis=1)
)

train_feat["sign_persistence"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

count    672072.000000
mean          0.411800
std           0.143634
min           0.018868
25%           0.320755
50%           0.433962
75%           0.509434
90%           0.566038
95%           0.584906
99%           1.000000
max           1.000000
Name: sign_persistence, dtype: float64

In [32]:
# Cortes aprendidos SOLO con train
persistence_bins = train_feat["sign_persistence"].quantile([0, .2, .4, .6, .8, 1]
).values
persistence_bins[0] = -np.inf
persistence_bins[-1] = np.inf

train_feat["persistence_quintile"] = pd.cut(train_feat["sign_persistence"],bins=persistence_bins,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"],
    include_lowest=True
)

val_feat["persistence_quintile"] = pd.cut(
    val_feat["sign_persistence"],
    bins=persistence_bins,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"],
    include_lowest=True
)

pd.concat({
    "train": train_feat.groupby(
        "persistence_quintile", observed=True
    )["directional"].mean(),

    "validation": val_feat.groupby(
        "persistence_quintile", observed=True
    )["directional"].mean()
}, axis=1)

,train,validation
persistence_quintile,,
Q1,0.481316,0.472737
Q2,0.605896,0.590638
Q3,0.627355,0.607277
Q4,0.633589,0.622646
Q5,0.614078,0.599635


hay una separación muy clara: las trayectorias con muy baja persistencia de signo (Q1) tienen mucha menor probabilidad de ser direccionales que Q2–Q5.

Y lo más importante: el patrón se replica en validation. No parece ser simplemente una peculiaridad del train.

La interpretación cuantitativa sería:

La persistencia de signo contiene información sobre la estructura de la trayectoria, especialmente para identificar caminos poco consistentes con un régimen direccional.

### reversal_intensity

- La idea es medir qué proporción de las transiciones entre retornos cambia de signo. 
- A diferencia de sign_persistence, aquí no importa hacia dónde termina la trayectoria; importa qué tan zigzagueante fue.

In [33]:
# Reversal intensity

def reversal_intensity(row):
    x = row.dropna().to_numpy()

    # Ignoramos retornos exactamente iguales a cero
    signs = np.sign(x)
    signs = signs[signs != 0]

    if len(signs) < 2:
        return np.nan

    reversals = np.sum(signs[1:] != signs[:-1])

    return reversals / (len(signs) - 1)


train_feat["reversal_intensity"] = (
    train_feat[return_cols].apply(reversal_intensity, axis=1)
)

val_feat["reversal_intensity"] = (
    val_feat[return_cols].apply(reversal_intensity, axis=1)
)

train_feat["reversal_intensity"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

count    657065.000000
mean          0.503598
std           0.120254
min           0.000000
25%           0.444444
50%           0.500000
75%           0.565217
90%           0.625000
95%           0.666667
99%           0.875000
max           1.000000
Name: reversal_intensity, dtype: float64

In [34]:
# Quintiles definidos SOLO con train
reversal_bins = train_feat["reversal_intensity"].quantile(
    [0, .2, .4, .6, .8, 1]
).values

reversal_bins[0] = -np.inf
reversal_bins[-1] = np.inf

train_feat["reversal_quintile"] = pd.cut(
    train_feat["reversal_intensity"],
    bins=reversal_bins,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"],
    include_lowest=True
)

val_feat["reversal_quintile"] = pd.cut(
    val_feat["reversal_intensity"],
    bins=reversal_bins,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"],
    include_lowest=True
)

pd.concat({
    "train": train_feat.groupby(
        "reversal_quintile", observed=True
    )["directional"].mean(),

    "validation": val_feat.groupby(
        "reversal_quintile", observed=True
    )["directional"].mean()
}, axis=1)

,train,validation
reversal_quintile,,
Q1,0.574407,0.562541
Q2,0.639656,0.626787
Q3,0.612877,0.592757
Q4,0.625853,0.611006
Q5,0.564761,0.559633


- NO entra

- el fenómeno es real, pero no dice simplemente “más reversals → menos directional”. Más bien aparece una forma de U invertida: tanto muy pocos como demasiados cambios de signo tienen menor probabilidad direccional, mientras niveles intermedios tienen mayor probabilidad.

### path_asymmetry.

- La actividad de la trayectoria está concentrada más en la primera mitad o en la segunda?

Para no mezclarla con dirección positiva/negativa, mediremos magnitud absoluta. Primero construimos la feature y vemos su distribución

In [35]:
# Path asymmetry

mid = len(return_cols) // 2

first_half_cols = return_cols[:mid]
second_half_cols = return_cols[mid:]

# Actividad absoluta de cada mitad
first_activity_train = train_feat[first_half_cols].abs().sum(axis=1)
second_activity_train = train_feat[second_half_cols].abs().sum(axis=1)

first_activity_val = val_feat[first_half_cols].abs().sum(axis=1)
second_activity_val = val_feat[second_half_cols].abs().sum(axis=1)

# Asimetría normalizada [-1, 1]
train_feat["path_asymmetry"] = (
    (second_activity_train - first_activity_train)
    / (second_activity_train + first_activity_train)
)

val_feat["path_asymmetry"] = (
    (second_activity_val - first_activity_val)
    / (second_activity_val + first_activity_val)
)

train_feat["path_asymmetry"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

count    664521.000000
mean         -0.273941
std           0.279130
min          -1.000000
25%          -0.412899
50%          -0.282829
75%          -0.148497
90%           0.001387
95%           0.137072
99%           0.904574
max           1.000000
Name: path_asymmetry, dtype: float64

In [36]:
# Quintiles definidos SOLO con train
asym_bins = train_feat["path_asymmetry"].quantile(
    [0, .2, .4, .6, .8, 1]
).values

asym_bins[0] = -np.inf
asym_bins[-1] = np.inf

train_feat["asym_quintile"] = pd.cut(
    train_feat["path_asymmetry"],
    bins=asym_bins,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"],
    include_lowest=True
)

val_feat["asym_quintile"] = pd.cut(
    val_feat["path_asymmetry"],
    bins=asym_bins,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"],
    include_lowest=True
)

pd.concat({
    "train": train_feat.groupby(
        "asym_quintile", observed=True
    )["directional"].mean(),

    "validation": val_feat.groupby(
        "asym_quintile", observed=True
    )["directional"].mean()
}, axis=1)

,train,validation
asym_quintile,,
Q1,0.516948,0.497424
Q2,0.611253,0.594397
Q3,0.627867,0.617433
Q4,0.641275,0.633413
Q5,0.591931,0.584841


La relación es no lineal: las trayectorias extremadamente cargadas hacia el inicio (Q1) son bastante menos direccionales; conforme la actividad se distribuye más hacia la segunda mitad, aumenta la probabilidad direccional hasta Q4; en Q5 vuelve a caer.

In [37]:
# ============================================================
# Final Quant Feature Builder
# ============================================================

def build_quant_features(
    train_df,
    val_df,
    return_cols,
    lower_q=0.001,
    upper_q=0.999
):
    """
    Construye las features cuantitativas finales.

    Los límites de winsorización se estiman exclusivamente
    utilizando train y después se aplican sin recalibración
    sobre validation.
    """

    train_out = train_df.copy()
    val_out = val_df.copy()

    # --------------------------------------------------------
    # 1. Winsorización por intervalo aprendida sólo en train
    # --------------------------------------------------------

    lower_bounds = train_df[return_cols].quantile(lower_q)
    upper_bounds = train_df[return_cols].quantile(upper_q)

    train_out[return_cols] = train_out[return_cols].clip(
        lower=lower_bounds,
        upper=upper_bounds,
        axis=1
    )

    val_out[return_cols] = val_out[return_cols].clip(
        lower=lower_bounds,
        upper=upper_bounds,
        axis=1
    )

    # Ventanas temporales
    early_cols = [f"r{i}" for i in range(0, 18)]
    middle_cols = [f"r{i}" for i in range(18, 36)]
    late_cols = [f"r{i}" for i in range(36, 53)]

    first_half_cols = return_cols[:len(return_cols) // 2]
    second_half_cols = return_cols[len(return_cols) // 2:]

    # --------------------------------------------------------
    # 2. Feature engineering
    # --------------------------------------------------------

    for df in [train_out, val_out]:

        # Disponibilidad
        df["n_obs"] = df[return_cols].notna().sum(axis=1)

        # ---- Path / displacement ----

        df["path_return_bps"] = (
            df[return_cols].sum(axis=1, skipna=True)
        )

        df["abs_path_return_bps"] = (
            df["path_return_bps"].abs()
        )

        df["return_early_bps"] = (
            df[early_cols].sum(axis=1, skipna=True)
        )

        df["return_middle_bps"] = (
            df[middle_cols].sum(axis=1, skipna=True)
        )

        df["return_late_bps"] = (
            df[late_cols].sum(axis=1, skipna=True)
        )

        df["early_late_change_bps"] = (
            df["return_late_bps"]
            - df["return_early_bps"]
        )

        # ---- Volatilidad ----

        squared_returns = df[return_cols] ** 2
        total_variation = squared_returns.sum(axis=1, skipna=True)

        df["realized_vol_bps"] = np.sqrt(total_variation)

        # ---- Path efficiency ----

        total_abs_move = (
            df[return_cols]
            .abs()
            .sum(axis=1, skipna=True)
        )

        df["path_efficiency"] = np.where(
            total_abs_move > 0,
            df["abs_path_return_bps"] / total_abs_move,
            0.0
        )

        # ---- Distribución temporal de volatilidad ----

        early_variation = (
            squared_returns[early_cols]
            .sum(axis=1, skipna=True)
        )

        late_variation = (
            squared_returns[late_cols]
            .sum(axis=1, skipna=True)
        )

        df["early_vol_share"] = np.where(
            total_variation > 0,
            early_variation / total_variation,
            0.0
        )

        df["late_vol_share"] = np.where(
            total_variation > 0,
            late_variation / total_variation,
            0.0
        )

        # ---- Concentración de volatilidad ----

        df["vol_concentration"] = np.where(
            total_variation > 0,
            squared_returns.max(axis=1) / total_variation,
            0.0
        )

        # ---- Persistencia de signo ----

        path_sign = np.sign(df["path_return_bps"])
        signs = np.sign(df[return_cols])

        observed = df[return_cols].notna()
        same_sign = signs.eq(path_sign, axis=0)

        df["sign_persistence"] = np.where(
            df["n_obs"] > 0,
            (same_sign & observed).sum(axis=1) / df["n_obs"],
            np.nan
        )

        # ---- Asimetría temporal ----

        first_activity = (
            df[first_half_cols]
            .abs()
            .sum(axis=1, skipna=True)
        )

        second_activity = (
            df[second_half_cols]
            .abs()
            .sum(axis=1, skipna=True)
        )

        total_activity = first_activity + second_activity

        df["path_asymmetry"] = np.where(
            total_activity > 0,
            (second_activity - first_activity) / total_activity,
            np.nan
        )

    return train_out, val_out, lower_bounds, upper_bounds

In [38]:
quant_features = [
    # Path
    "path_return_bps",
    "abs_path_return_bps",
    "return_early_bps",
    "return_middle_bps",
    "return_late_bps",
    "early_late_change_bps",

    # Activity / volatility
    "realized_vol_bps",

    # Path structure
    "path_efficiency",
    "early_vol_share",
    "late_vol_share",
    "vol_concentration",
    "sign_persistence",
    "path_asymmetry",

    # Data availability
    "n_obs",
]

In [39]:
train_features, val_features, lower_bounds, upper_bounds = (
    build_quant_features(
        train_dev,
        val_dev,
        return_cols
    )
)

print("Train:", train_features.shape)
print("Validation:", val_features.shape)

train_features[quant_features].describe().T

Train: (673751, 71)
Validation: (169548, 71)


,count,mean,std,min,25%,50%,75%,max
path_return_bps,673751.0,-33.468035,292.186858,-3491.968090,-107.520000,-6.330000,79.350000,4185.643130
abs_path_return_bps,673751.0,164.155974,244.020587,0.000000,38.640000,92.700000,191.230000,4185.643130
return_early_bps,673751.0,-29.811015,265.006347,-3437.916770,-81.540000,-1.970000,61.530000,3000.455300
return_middle_bps,673751.0,-2.373711,92.840922,-1552.372450,-41.100000,0.000000,37.570000,2040.337300
return_late_bps,673751.0,-1.283308,71.961493,-1124.357350,-30.570000,0.000000,28.700000,1546.852920
early_late_change_bps,673751.0,28.527707,273.466402,-3097.847610,-70.160000,2.680000,88.850000,3571.486090
realized_vol_bps,673751.0,194.379075,216.936893,0.000000,93.029961,140.393075,219.068160,2627.706886
path_efficiency,673751.0,0.219727,0.211927,0.000000,0.073092,0.161240,0.289486,1.000000
early_vol_share,673751.0,0.612101,0.222171,0.000000,0.493005,0.638008,0.763056,1.000000
late_vol_share,673751.0,0.148748,0.139067,0.000000,0.063303,0.116643,0.192600,1.000000
